# Crime Category and Offense Composition

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from crime_snapshot import load_crime_snapshot
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

df, metadata = load_crime_snapshot(
    PROJECT_ROOT / "data" / "processed" / "crime"
)

display(df.head())

,report_number,report_date_time,offense_id,offense_date,nibrs_group_a_b,nibrs_crime_against_category,offense_sub_category,shooting_type_group,block_address,latitude,longitude,beat,precinct,sector,neighborhood,reporting_area,offense_category,nibrs_offense_code_description,nibrs_offense_code,census_block_2020
0,2025-219588,2025-08-03 11:56:33,65248246180,2025-08-01 20:00:00,a,person,aggravated assault,-,6XX BLOCK OF BROADWAY,47.607646,-122.320754,e2,east,e,first hill,12278,violent crime,aggravated assault,13A,8600.3001
1,2025-219588,2025-08-03 11:56:33,65328828453,2025-08-01 20:00:00,b,not_a_crime,999,-,6XX BLOCK OF BROADWAY,47.607646,-122.320754,e2,east,e,first hill,12278,all other,not reportable to nibrs,999,8600.3001
2,2025-221688,2025-08-04 16:01:21,65260476770,2025-08-01 20:00:00,a,property,larceny-theft,-,1XX BLOCK OF 6TH AVE S,47.601290,-122.326352,k3,west,k,chinatown/international district,822,property crime,all other larceny,23H,9200.1003
3,2025-221520,2025-08-04 16:16:03,65260591286,2025-08-01 20:21:00,a,person,assault offenses,-,25XX BLOCK OF ALKI AVE SW,47.581069,-122.405679,w1,southwest,w,alki,8453,all other,simple assault,13B,9701.3017
4,2025-220900,2025-08-03 20:23:59,65343613386,2025-08-01 20:30:00,a,property,larceny-theft,-,17XX BLOCK OF 18TH AVE,47.617023,-122.308839,c2,east,c,central area/squire park,7563,property crime,theft of motor vehicle parts or accessories,23G,7901.1005


In [2]:
required_columns = [
    "offense_id",
    "offense_date",
    "report_number",
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "shooting_type_group",
    "neighborhood",
    "precinct",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

missing_columns

[]

In [3]:
classification_columns = [
    "offense_category",
    "offense_sub_category",
    "nibrs_crime_against_category",
    "nibrs_group_a_b",
    "nibrs_offense_code_description",
    "nibrs_offense_code",
    "shooting_type_group",
]

category_missingness = (
    df[classification_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

category_missingness

offense_category                  0.0
offense_sub_category              0.0
nibrs_crime_against_category      0.0
nibrs_group_a_b                   0.0
nibrs_offense_code_description    0.0
nibrs_offense_code                0.0
shooting_type_group               0.0
dtype: float64

In [4]:
offense_category_summary = (
    df
    .groupby("offense_category", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

offense_category_summary["offense_share_percent"] = (
    offense_category_summary["offense_count"]
    / offense_category_summary["offense_count"].sum()
    * 100
).round(2)

offense_category_summary

fig = px.bar(
    offense_category_summary.head(20),
    x="offense_category",
    y="offense_count",
    title="Top Crime Categories",
    template="plotly_dark",
)

fig.update_layout(
    xaxis_title="Offense Category",
    yaxis_title="Reported Offenses",
    xaxis_tickangle=-45,
)

fig.show()

In [5]:
offense_sub_category_summary = (
    df
    .groupby("offense_sub_category", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

offense_sub_category_summary["offense_share_percent"] = (
    offense_sub_category_summary["offense_count"]
    / offense_sub_category_summary["offense_count"].sum()
    * 100
).round(2)

offense_sub_category_summary.head(30)

fig = px.bar(
    offense_sub_category_summary.head(30),
    x="offense_sub_category",
    y="offense_count",
    title="Top Crime Sub-Categories",
    template="plotly_dark",
)

fig.update_layout(
    xaxis_title="Offense Sub-Category",
    yaxis_title="Reported Offenses",
    xaxis_tickangle=-45,
)

fig.show()

In [12]:
category_to_subcategory = (
    df
    .groupby(["offense_category", "offense_sub_category"], dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
    .sort_values(
        ["offense_category", "offense_count"],
        ascending=[True, False],
    )
)

category_to_subcategory

top_subcategories_by_category = (
    category_to_subcategory
    .groupby("offense_category", group_keys=False)
)

top_subcategories_by_category

In [7]:
category_vs_nibrs = (
    df
    .groupby(
        ["offense_category", "nibrs_crime_against_category"],
        dropna=False,
    )
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

category_vs_nibrs

category_vs_nibrs_pivot = (
    category_vs_nibrs
    .pivot_table(
        index="offense_category",
        columns="nibrs_crime_against_category",
        values="offense_count",
        fill_value=0,
    )
)

category_vs_nibrs_pivot

nibrs_crime_against_category,-,any,not_a_crime,person,property,society
offense_category,,,,,,
all other,37.0,2597.0,9038.0,8563.0,9791.0,6516.0
property crime,0.0,0.0,0.0,0.0,35416.0,0.0
violent crime,0.0,0.0,0.0,3621.0,1440.0,0.0


In [8]:
category_share = (
    df["offense_category"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

rare_category_share = category_share[category_share < 1]

rare_category_share

Series([], Name: proportion, dtype: double[pyarrow])

In [9]:
neighborhood_category_counts = (
    df
    .dropna(subset=["neighborhood"])
    .groupby(["neighborhood", "offense_category"], dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
    )
    .reset_index()
)

neighborhood_totals = (
    neighborhood_category_counts
    .groupby("neighborhood")["offense_count"]
    .sum()
    .reset_index(name="neighborhood_total")
)

neighborhood_category_counts = neighborhood_category_counts.merge(
    neighborhood_totals,
    on="neighborhood",
    how="left",
)

neighborhood_category_counts["category_share_percent"] = (
    neighborhood_category_counts["offense_count"]
    / neighborhood_category_counts["neighborhood_total"]
    * 100
).round(2)

neighborhood_category_counts.sort_values(
    ["neighborhood_total", "offense_count"],
    ascending=[False, False],
)

,neighborhood,offense_category,offense_count,neighborhood_total,category_share_percent
24,capitol hill,all other,3300,6244,52.85
25,capitol hill,property crime,2479,6244,39.7
26,capitol hill,violent crime,465,6244,7.45
140,queen anne,property crime,2687,4466,60.17
139,queen anne,all other,1592,4466,35.65
...,...,...,...,...,...
135,pigeon point,violent crime,1,69,1.45
42,commercial harbor island,all other,19,33,57.58
43,commercial harbor island,property crime,13,33,39.39
44,commercial harbor island,violent crime,1,33,3.03


In [10]:
shooting_summary = (
    df
    .groupby("shooting_type_group", dropna=False)
    .agg(
        offense_count=("offense_id", "size"),
        unique_reports=("report_number", "nunique"),
    )
    .reset_index()
    .sort_values("offense_count", ascending=False)
)

shooting_summary

,shooting_type_group,offense_count,unique_reports
0,-,76254,64316
3,shots fired (eyewitness/casings/property damage),598,421
2,shooting (non-fatal injury),134,88
1,shooting (fatal injury),33,21


In [36]:
top_subcategories_by_category = (
    category_to_subcategory.groupby("offense_category")["offense_sub_category"]
    .value_counts()
    .reset_index(name="count")
)

top_subcategories_by_category[top_subcategories_by_category['offense_category'] == 'all other']


,offense_category,offense_sub_category,count
0,all other,999,1
1,all other,assault offenses,1
2,all other,"property offenses (includes stolen, destruction)",1
3,all other,extortion/fraud/forgery/bribery (includes bad ...,1
4,all other,all other,1
5,all other,narcotic violations (includes drug equip.),1
6,all other,trespass,1
7,all other,violation of no contact order,1
8,all other,dui,1
9,all other,weapon law violation,1
